# 3. HMM State Discovery

Fit Hidden Markov Models to session-level summary statistics to discover
behavioural states in a data-driven manner. The goal: define "expert"
functionally as a latent state rather than by arbitrary session thresholds.

**Approach:**
1. Compute selected stats per session for each animal
2. Z-score and (optionally) PCA-reduce
3. Fit Gaussian HMMs with 2-6 states, select by BIC
4. Viterbi-decode state sequences
5. Interpret states by projecting back to original feature space
6. Validate: compare HMM states with true eta (synthetic data)

**Library:** Scott Linderman's SSM (install: `pip install ssm`)
or hmmlearn as fallback.

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import spearmanr
import warnings, time
warnings.filterwarnings('ignore')

from Analysis.summary_stats import compute_summary_stats, get_stat_names_expanded
from Data.structures import (
    generate_synthetic_animal, param_trajectory_naive_to_expert,
)

# Try SSM first, fall back to hmmlearn
try:
    import ssm
    HMM_BACKEND = 'ssm'
    print("Using SSM backend")
except ImportError:
    try:
        from hmmlearn import hmm as hmmlearn_hmm
        HMM_BACKEND = 'hmmlearn'
        print("Using hmmlearn backend (install ssm for full functionality)")
    except ImportError:
        HMM_BACKEND = None
        print("WARNING: Neither ssm nor hmmlearn installed.")
        print("Install one: pip install ssm  OR  pip install hmmlearn")

## 1. Generate Synthetic Data

Multiple animals with known eta trajectories. The HMM should recover
states that correspond to high-eta (naive/learning) vs low-eta (expert).

In [ ]:
# --- Configuration ---
N_ANIMALS = 8
N_SESSIONS = 30
N_TRIALS = 300
SEED = 42

# Stats to use (based on notebook 2 sensitivity analysis)
SELECTED_STATS = [
    'accuracy', 'psychometric', 'recency', 'win_stay',
    'stimulus_sensitivity', 'choice_entropy',
    'logistic_history',
]

In [ ]:
# --- Generate animals with varied trajectories ---
rng_master = np.random.default_rng(SEED)
expanded_names = get_stat_names_expanded(SELECTED_STATS)
n_features = len(expanded_names)

animals_data = []  # List of dicts with stats, eta, etc.

print(f"Generating {N_ANIMALS} animals...")
for a in range(N_ANIMALS):
    animal_seed = int(rng_master.integers(0, 2**31))
    
    eta_start = rng_master.uniform(0.35, 0.55)
    eta_end = rng_master.uniform(0.06, 0.12)
    decay_rate = rng_master.uniform(0.08, 0.20)
    sigma_percep = rng_master.uniform(0.10, 0.20)
    A_repulsion = rng_master.uniform(0.05, 0.18)
    
    params = param_trajectory_naive_to_expert(
        N_SESSIONS, eta_start=eta_start, eta_end=eta_end,
        decay_rate=decay_rate, sigma_percep=sigma_percep,
        A_repulsion=A_repulsion,
    )
    
    animal, gt = generate_synthetic_animal(
        animal_id=f'SYN_{a:02d}', true_params=params,
        trials_per_session=N_TRIALS, seed=animal_seed,
    )
    
    # Compute stats per session
    stat_matrix = np.empty((N_SESSIONS, n_features))
    for s_idx, session in enumerate(animal.sessions):
        arrays = session.trials.get_model_arrays(exclude_abort=True)
        try:
            stats = compute_summary_stats(
                arrays['choices'], arrays['stimuli'], arrays['categories'],
                stat_names=SELECTED_STATS, return_dict=False,
            )
            stat_matrix[s_idx] = stats if len(stats) == n_features else np.nan
        except Exception:
            stat_matrix[s_idx] = np.nan
    
    animals_data.append({
        'animal_id': f'SYN_{a:02d}',
        'stat_matrix': stat_matrix,
        'eta': np.array(gt['params_per_session']['eta_learning']),
        'session_idx': np.arange(N_SESSIONS),
    })
    print(f"  {a}: eta {eta_start:.2f}->{eta_end:.2f}, decay={decay_rate:.2f}")

print("Done.")

## 2. Preprocessing: Z-score and PCA

In [ ]:
# --- Stack all sessions, z-score, PCA ---
all_stats = np.vstack([ad['stat_matrix'] for ad in animals_data])
all_eta = np.concatenate([ad['eta'] for ad in animals_data])
all_animal_idx = np.concatenate([np.full(N_SESSIONS, a) for a in range(N_ANIMALS)])
all_session_idx = np.concatenate([ad['session_idx'] for ad in animals_data])

# Handle NaNs: replace with column mean
col_means = np.nanmean(all_stats, axis=0)
nan_mask = np.isnan(all_stats)
for j in range(n_features):
    all_stats[nan_mask[:, j], j] = col_means[j]

# Z-score
scaler = StandardScaler()
all_stats_z = scaler.fit_transform(all_stats)

# PCA
pca = PCA()
all_stats_pca = pca.fit_transform(all_stats_z)
cum_var = np.cumsum(pca.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.bar(range(len(cum_var)), pca.explained_variance_ratio_, color='#1976d2', alpha=0.7)
ax.plot(range(len(cum_var)), cum_var, 'ko-', markersize=4)
ax.set_xlabel('PC')
ax.set_ylabel('Variance explained')
ax.set_title('PCA of summary stats')
ax.axhline(0.9, color='grey', linestyle='--', linewidth=0.5)

# How many PCs for 90% variance?
n_pcs_90 = np.searchsorted(cum_var, 0.9) + 1
print(f"PCs for 90% variance: {n_pcs_90}")

# Loadings of first 3 PCs
ax = axes[1]
for pc in range(min(3, n_features)):
    ax.barh(np.arange(n_features) + pc * 0.25, pca.components_[pc],
            height=0.25, alpha=0.7, label=f'PC{pc+1}')
ax.set_yticks(range(n_features))
ax.set_yticklabels(expanded_names, fontsize=7)
ax.set_xlabel('Loading')
ax.set_title('PCA loadings')
ax.legend(fontsize=8)
ax.invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# --- Choose number of PCs to use ---
# Use enough PCs for 90% variance, minimum 3
N_PCS = max(3, n_pcs_90)
print(f"Using {N_PCS} PCs ({cum_var[N_PCS-1]:.1%} variance)")

# Split back into per-animal sequences for HMM
obs_per_animal = []
for a in range(N_ANIMALS):
    start = a * N_SESSIONS
    end = start + N_SESSIONS
    obs_per_animal.append(all_stats_pca[start:end, :N_PCS])

## 3. Fit HMMs with Model Selection

In [ ]:
# --- Fit HMMs with 2-6 states, compare BIC ---
K_range = range(2, 7)
results = {}

if HMM_BACKEND == 'ssm':
    print("Fitting with SSM...")
    for K in K_range:
        # Fit pooled HMM: all animals share parameters
        model = ssm.HMM(K, N_PCS, observations="gaussian")
        
        # SSM expects list of observation arrays
        lls = model.fit(obs_per_animal, method="em", num_iters=200, verbose=0)
        
        # Compute log-likelihood and BIC
        total_ll = sum(model.log_probability(obs) for obs in obs_per_animal)
        n_obs = sum(len(obs) for obs in obs_per_animal)
        # Number of params: K*K transition + K*D means + K*D*(D+1)/2 covariances + K-1 initial
        n_params = K*K + K*N_PCS + K*N_PCS*(N_PCS+1)//2 + (K-1)
        bic = -2 * total_ll + n_params * np.log(n_obs)
        
        # Viterbi decode
        states_per_animal = [model.most_likely_states(obs) for obs in obs_per_animal]
        
        results[K] = {
            'model': model, 'bic': bic, 'll': total_ll,
            'states': states_per_animal, 'n_params': n_params,
        }
        print(f"  K={K}: LL={total_ll:.1f}, BIC={bic:.1f}")

elif HMM_BACKEND == 'hmmlearn':
    print("Fitting with hmmlearn...")
    # Concatenate all observations with lengths
    obs_concat = np.vstack(obs_per_animal)
    lengths = [len(obs) for obs in obs_per_animal]
    
    for K in K_range:
        best_ll = -np.inf
        best_model = None
        
        # Multiple restarts for robustness
        for restart in range(5):
            model = hmmlearn_hmm.GaussianHMM(
                n_components=K, covariance_type='full',
                n_iter=200, random_state=restart,
            )
            try:
                model.fit(obs_concat, lengths)
                ll = model.score(obs_concat, lengths)
                if ll > best_ll:
                    best_ll = ll
                    best_model = model
            except Exception:
                continue
        
        if best_model is None:
            print(f"  K={K}: all restarts failed")
            continue
        
        n_obs = len(obs_concat)
        n_params = K*K + K*N_PCS + K*N_PCS*(N_PCS+1)//2 + (K-1)
        bic = -2 * best_ll + n_params * np.log(n_obs)
        
        # Decode per animal
        states_per_animal = []
        offset = 0
        for length in lengths:
            chunk = obs_concat[offset:offset+length]
            states_per_animal.append(best_model.predict(chunk))
            offset += length
        
        results[K] = {
            'model': best_model, 'bic': bic, 'll': best_ll,
            'states': states_per_animal, 'n_params': n_params,
        }
        print(f"  K={K}: LL={best_ll:.1f}, BIC={bic:.1f}")

else:
    print("No HMM backend available. Install ssm or hmmlearn.")

In [ ]:
# --- Model selection plot ---
if results:
    Ks = sorted(results.keys())
    bics = [results[k]['bic'] for k in Ks]
    lls = [results[k]['ll'] for k in Ks]
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    
    axes[0].plot(Ks, bics, 'ko-', markersize=8)
    best_K = Ks[np.argmin(bics)]
    axes[0].scatter([best_K], [min(bics)], c='red', s=100, zorder=3)
    axes[0].set_xlabel('Number of states')
    axes[0].set_ylabel('BIC')
    axes[0].set_title(f'Model selection (best K={best_K})')
    
    axes[1].plot(Ks, lls, 'ko-', markersize=8)
    axes[1].set_xlabel('Number of states')
    axes[1].set_ylabel('Log-likelihood')
    axes[1].set_title('Log-likelihood')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nBest K by BIC: {best_K}")

## 4. Interpret HMM States

In [ ]:
# --- Visualise state sequences alongside true eta ---
if results:
    best = results[best_K]
    
    fig, axes = plt.subplots(N_ANIMALS, 1, figsize=(14, 2.5 * N_ANIMALS), sharex=True)
    if N_ANIMALS == 1:
        axes = [axes]
    
    cmap = plt.cm.Set1
    
    for a, ax in enumerate(axes):
        states = best['states'][a]
        eta = animals_data[a]['eta']
        sess = animals_data[a]['session_idx']
        
        # Colour background by state
        for s_idx in range(len(sess)):
            ax.axvspan(s_idx - 0.5, s_idx + 0.5,
                       color=cmap(states[s_idx] / max(best_K - 1, 1)),
                       alpha=0.3)
        
        # Plot eta
        ax.plot(sess, eta, 'k-o', markersize=4, linewidth=1.5, label='true eta')
        ax.set_ylabel(f'A{a}', fontsize=9)
        ax.set_ylim(0, 0.6)
        if a == 0:
            ax.set_title(f'HMM states (K={best_K}) vs true eta')
            # Legend for states
            from matplotlib.patches import Patch
            patches = [Patch(facecolor=cmap(k / max(best_K-1, 1)), alpha=0.3,
                            label=f'State {k}') for k in range(best_K)]
            ax.legend(handles=patches + [plt.Line2D([0],[0], color='k', marker='o',
                      label='true eta')], fontsize=7, ncol=best_K+1, loc='upper right')
    
    axes[-1].set_xlabel('Session')
    plt.tight_layout()
    plt.show()

In [ ]:
# --- State means in original feature space ---
if results:
    best = results[best_K]
    all_states = np.concatenate(best['states'])
    
    # Compute mean of z-scored features per state
    state_means_z = np.empty((best_K, n_features))
    state_means_eta = np.empty(best_K)
    
    for k in range(best_K):
        mask = all_states == k
        state_means_z[k] = all_stats_z[mask].mean(axis=0)
        state_means_eta[k] = all_eta[mask].mean()
    
    # Sort states by mean eta (so state 0 = lowest eta = most expert)
    state_order = np.argsort(state_means_eta)
    
    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(n_features)
    width = 0.8 / best_K
    
    for i, k in enumerate(state_order):
        offset = (i - best_K / 2 + 0.5) * width
        ax.bar(x + offset, state_means_z[k], width, alpha=0.7,
               color=cmap(k / max(best_K-1, 1)),
               label=f'State {k} (mean eta={state_means_eta[k]:.3f})')
    
    ax.set_xticks(x)
    ax.set_xticklabels(expanded_names, rotation=90, fontsize=7)
    ax.set_ylabel('Z-scored feature mean')
    ax.set_title('HMM state profiles in original feature space')
    ax.legend(fontsize=8)
    ax.axhline(0, color='k', linewidth=0.5)
    plt.tight_layout()
    plt.show()

## 5. Validation: HMM States vs True eta

In [ ]:
# --- Quantitative validation ---
if results:
    best = results[best_K]
    all_states = np.concatenate(best['states'])
    
    # Box plot: eta distribution per state
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    ax = axes[0]
    state_etas = [all_eta[all_states == k] for k in range(best_K)]
    bp = ax.boxplot(state_etas, labels=[f'State {k}' for k in range(best_K)],
                    patch_artist=True)
    for i, patch in enumerate(bp['boxes']):
        patch.set_facecolor(cmap(i / max(best_K-1, 1)))
        patch.set_alpha(0.5)
    ax.set_ylabel('True eta_learning')
    ax.set_title('eta distribution per HMM state')
    
    # Transition matrix
    ax = axes[1]
    if HMM_BACKEND == 'ssm':
        T = np.exp(best['model'].transitions.log_Ps)
    else:
        T = best['model'].transmat_
    
    im = ax.imshow(T, cmap='Blues', vmin=0, vmax=1)
    for i in range(best_K):
        for j in range(best_K):
            ax.text(j, i, f'{T[i,j]:.2f}', ha='center', va='center', fontsize=10)
    ax.set_xlabel('To state')
    ax.set_ylabel('From state')
    ax.set_title('Transition matrix')
    plt.colorbar(im, ax=ax, shrink=0.8)
    
    plt.tight_layout()
    plt.show()
    
    # Spearman correlation between state assignment and eta
    rho, pval = spearmanr(all_states, all_eta)
    print(f"Spearman correlation (state vs eta): rho={rho:.3f}, p={pval:.2e}")
    print(f"(Sign depends on arbitrary state labelling)")

## 6. Online Classification Mode

Once the HMM is fitted, classify new sessions incrementally.
This is how you'd use it in practice: given a new animal's session,
ask "which state is this animal most likely in?" 

In [ ]:
# --- Online classification demo ---
if results:
    best = results[best_K]
    model = best['model']
    
    # Generate a new test animal
    test_params = param_trajectory_naive_to_expert(
        N_SESSIONS, eta_start=0.42, eta_end=0.09, decay_rate=0.14
    )
    test_animal, test_gt = generate_synthetic_animal(
        animal_id='SYN_TEST', true_params=test_params,
        trials_per_session=N_TRIALS, seed=999,
    )
    
    # Compute stats, transform, classify
    test_stats = np.empty((N_SESSIONS, n_features))
    for s_idx, session in enumerate(test_animal.sessions):
        arrays = session.trials.get_model_arrays(exclude_abort=True)
        try:
            stats = compute_summary_stats(
                arrays['choices'], arrays['stimuli'], arrays['categories'],
                stat_names=SELECTED_STATS, return_dict=False,
            )
            test_stats[s_idx] = stats if len(stats) == n_features else col_means
        except Exception:
            test_stats[s_idx] = col_means
    
    test_z = scaler.transform(test_stats)
    test_pca = pca.transform(test_z)[:, :N_PCS]
    
    if HMM_BACKEND == 'ssm':
        test_states = model.most_likely_states(test_pca)
    else:
        test_states = model.predict(test_pca)
    
    # Plot
    fig, ax = plt.subplots(figsize=(12, 4))
    sess = np.arange(N_SESSIONS)
    eta_test = test_gt['params_per_session']['eta_learning']
    
    for s_idx in range(N_SESSIONS):
        ax.axvspan(s_idx - 0.5, s_idx + 0.5,
                   color=cmap(test_states[s_idx] / max(best_K-1, 1)), alpha=0.3)
    
    ax.plot(sess, eta_test, 'k-o', markersize=5, linewidth=2, label='true eta')
    ax.set_xlabel('Session')
    ax.set_ylabel('eta')
    ax.set_title('Online classification of new animal')
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    # When does the HMM say this animal became expert?
    expert_state = state_order[0]  # State with lowest mean eta
    expert_sessions = np.where(test_states == expert_state)[0]
    if len(expert_sessions) > 0:
        # First run of consecutive expert sessions
        diffs = np.diff(expert_sessions)
        runs_start = [expert_sessions[0]]
        for i, d in enumerate(diffs):
            if d > 1:
                runs_start.append(expert_sessions[i+1])
        print(f"Expert state first reached at session {expert_sessions[0]}")
        print(f"True eta at that session: {eta_test[expert_sessions[0]]:.3f}")
    else:
        print("Expert state never reached in this animal")

---
**Next:** `4_sbi_inference.ipynb` — Fit the BE model to behavioural data
using Simulation-Based Inference with the validated summary statistics.